# MorphIQ SDK for Python — books.revalue

Edit the synthetic inputs below or paste your own JSON. Setup and input edits send **zero requests**.
Quote sends one utility request. Execute sends one calculation with the entered CU ceiling,
consumes the quote, and never retries. Input edits invalidate the quote and current result;
receipts remain in memory. Credentials come from the environment or a masked prompt when an action is clicked.

Option P&L uses the base model mark. Book revaluation and explanation use base model value;
book solves use signed entry value (a debit is positive, a credit negative).
Overlapping strategy totals must not be added together. All positions use one declared currency;
no FX conversion is performed. Solver intervals are sampled evidence, and unresolved coverage
is not proof of no solution. Quote the actual request instead of inferring price from work counts.

[Start Here](../start-here.ipynb)


In [ ]:
import json
import os
from getpass import getpass

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from morphiq_sdk import APIError, BooksRevalueRequest, MorphIQ, ResponseError, TransportError

inputs = widgets.Textarea(
    value="""
{
  "currency": "USD",
  "markets": {
    "SPY": {
      "spot": 100.0,
      "risk_free_rate": 0.05,
      "dividend_yield": 0.0
    }
  },
  "books": [
    {
      "id": "spread",
      "positions": [
        {
          "id": "long-call",
          "underlying": "SPY",
          "quantity": 1.0,
          "multiplier": 100.0,
          "option": {
            "type": "call",
            "exercise_style": "european",
            "strike": 100.0,
            "time_to_expiry_years": 1.0
          },
          "volatility": 0.2,
          "model": {
            "id": "black_scholes_merton"
          }
        },
        {
          "id": "short-call",
          "underlying": "SPY",
          "quantity": -1.0,
          "multiplier": 100.0,
          "option": {
            "type": "call",
            "exercise_style": "european",
            "strike": 110.0,
            "time_to_expiry_years": 1.0
          },
          "volatility": 0.2,
          "model": {
            "id": "black_scholes_merton"
          }
        }
      ],
      "strategies": [
        {
          "id": "vertical",
          "position_ids": [
            "long-call",
            "short-call"
          ]
        },
        {
          "id": "long-leg",
          "position_ids": [
            "long-call"
          ]
        }
      ]
    }
  ],
  "scenario": {
    "name": "up-and-steeper",
    "shocks": [
      {
        "kind": "spot_relative",
        "relative": 0.05
      },
      {
        "kind": "parallel_vol",
        "shift": 0.01
      }
    ]
  }
}
""",
    description="Inputs JSON:",
    layout=widgets.Layout(width="100%", height="340px"),
)
cap = widgets.Text(value="1000", description="Max CU:")
quote_button = widgets.Button(description="Quote (1 request)")
execute_button = widgets.Button(description="Execute (1 request)", disabled=True)
output = widgets.Output()
receipts = []
quote_state = None
last_response = None
busy = False


def invalidate(change):
    global quote_state, last_response
    quote_state = None
    last_response = None
    execute_button.disabled = True
    output.clear_output()


inputs.observe(invalidate, names="value")


def run_action(quote_only):
    global quote_state, last_response, busy
    if busy:
        return
    busy = True
    quote_button.disabled = execute_button.disabled = True
    try:
        request = BooksRevalueRequest.model_validate(json.loads(inputs.value))
        wire = request.to_wire()
        if not quote_only:
            if quote_state is None or quote_state[0] != wire:
                raise ValueError("Quote these exact inputs before executing.")
            if not cap.value.isascii() or not cap.value.isdecimal():
                raise ValueError("Max CU must be a nonnegative whole number.")
            maximum = int(cap.value)
            if not 0 <= maximum <= 2**64 - 1 or maximum < quote_state[1]:
                raise ValueError("Max CU must cover the quote and fit an unsigned 64-bit integer.")
        quote_state = None  # Consume before I/O, including timeouts. Never replay a solve.
        api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")
        with MorphIQ(api_key=api_key) as client:
            if quote_only:
                response = client.rate_card.quote(request)
                quote_state = (wire, response.quote.total_cu)
            else:
                response = client.books.revalue(request, max_cu=maximum)
                last_response = response
        charge = response.metadata.charged_cu
        receipts.append(
            {
                "action": "quote" if quote_only else "execute",
                "request_id": response.metadata.request_id,
                "charged_cu": charge if charge is not None else "not reported",
            }
        )
        with output:
            output.clear_output()
            if quote_only:
                print("Quoted CU:", response.quote.total_cu, "(quote, not a charge)")
            else:
                result = response.result.model_dump(mode="json", exclude_unset=True)
                # Keep the complete result, including uncertainty and any per-point problems.
                display(pd.json_normalize(result))
                print(json.dumps(result, indent=2))
            display(pd.DataFrame(receipts))
    except APIError as error:
        charge = error.metadata.charged_cu if error.metadata else None
        receipts.append(
            {
                "action": "error",
                "request_id": error.request_id,
                "charged_cu": charge if charge is not None else "not reported",
            }
        )
        with output:
            print(error.code, error.detail)
            display(pd.DataFrame(receipts))
    except (TransportError, ResponseError):
        quote_state = None
        receipts.append({"action": "unknown outcome", "charged_cu": "unknown"})
        with output:
            print("Execution and charge may be unknown. No retry was attempted.")
            display(pd.DataFrame(receipts))
    except ValueError as error:
        with output:
            print(str(error))
    finally:
        busy = False
        quote_button.disabled = False
        execute_button.disabled = quote_state is None


quote_button.on_click(lambda _: run_action(True))
execute_button.on_click(lambda _: run_action(False))
display(inputs, cap, widgets.HBox([quote_button, execute_button]), output)